# 第07回：AfDesignで固定骨格設計
目標はPDB **1TENの鎖A**。公式ColabDesignの固定骨格設計例を授業用に絞りました。

T4 GPUを選び「すべてのセルを実行」。演習20分を目安に、配列・損失・予測構造を観察します。初回の準備やGPU割り当てには別途時間がかかります。採点・提出はありません。

**設定**：soft 100 / temp 30 / hard 10、1モデル、recycle 0、seed 42。短い動作観察用の設定です。公式例の十分な最適化や、実験での成功を保証する条件ではありません。AF2の重みは更新せず、配列を最適化します。


In [ ]:
from pathlib import Path
import subprocess,sys,os,time,json,urllib.request
start=time.perf_counter()
work=Path('/content/bioinfo07');work.mkdir(exist_ok=True);os.chdir(work)
r=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
assert r.returncode==0,'ランタイムをT4 GPUへ変更してください。'
print(r.stdout)
subprocess.run([sys.executable,'-m','pip','-q','install','uv','py3Dmol==2.5.3'],check=True)
py=str(work/'.venv/bin/python')
if not Path(py).exists():subprocess.run(['uv','venv','--python','3.11',str(work/'.venv')],check=True)


In [ ]:
# 初回はGPU用ライブラリと学習済みパラメータを取得します。
subprocess.run(['uv','pip','install','--python',py,
 'numpy==1.26.4','scipy==1.12.0','jax[cuda12]==0.4.35','nvidia-cuda-nvcc-cu12==12.6.85','dm-haiku==0.0.13','optax==0.2.4','chex==0.1.88',
 'git+https://github.com/sokrypton/ColabDesign.git@e31a56fe1d9b4de25c8697f3a28b75892941cc72'],check=True)
params=work/'params';params.mkdir(exist_ok=True)
if not (params/'params_model_1_ptm.npz').exists():
    archive=work/'alphafold_params_2022-12-06.tar'
    subprocess.run(['wget','-q','-c','https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar','-O',str(archive)],check=True)
    import tarfile
    with tarfile.open(archive) as tar:
        member=next(m for m in tar.getmembers() if Path(m.name).name=='params_model_1_ptm.npz')
        with tar.extractfile(member) as src, open(params/'params_model_1_ptm.npz','wb') as dst:
            import shutil;shutil.copyfileobj(src,dst)
urllib.request.urlretrieve('https://files.rcsb.org/download/1TEN.pdb',work/'1TEN.pdb')
print('準備秒',round(time.perf_counter()-start))


## 配列を最適化する
別のPython環境で計算します。初回はJAXのコンパイルがあり、ログがしばらく止まって見える場合があります。
損失の値は同じ定義・条件の中で比較します。RMSDとpLDDTを同じ量として扱いません。

In [ ]:
script = "import os\nos.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'\nos.environ['MPLBACKEND']='Agg'\nimport json,time\nfrom pathlib import Path\nimport numpy as np\nimport jax\nimport matplotlib;matplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nfrom colabdesign import mk_afdesign_model\nassert any(d.platform=='gpu' for d in jax.devices()), 'GPUが使えません'\nprint('devices',jax.devices(),flush=True)\nt0=time.perf_counter()\nmodel=mk_afdesign_model(protocol='fixbb',data_dir='params',model_names=['model_1_ptm'],num_recycles=0)\nmodel.prep_inputs(pdb_filename='1TEN.pdb',chain='A')\nmodel.restart(seed=42)\nmodel.design_3stage(soft_iters=100,temp_iters=30,hard_iters=10,num_models=1)\nseq=model.get_seqs()[0]\nmodel.save_pdb('design.pdb')\nmodel.plot_traj();plt.savefig('trajectory.png',dpi=160,bbox_inches='tight');plt.close('all')\nreport={'sequence':seq,'length':len(seq),'seconds':time.perf_counter()-t0,'conditions':{'pdb':'1TEN','chain':'A','seed':42,'soft':100,'temp':30,'hard':10,'model':'model_1_ptm','num_recycles':0},'last_step_metrics':{k:float(np.asarray(v).reshape(-1)[0]) for k,v in model.aux['log'].items() if np.asarray(v).size==1}}\nPath('result.json').write_text(json.dumps(report,indent=2))\nPath('design.fasta').write_text('>bioinfo07_fixbb_1TEN\\n'+seq+'\\n')\nprint(json.dumps(report),flush=True)\n"
(work/'run_design.py').write_text(script)
process=subprocess.Popen([py,'run_design.py'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for line in process.stdout:print(line,end='',flush=True)
assert process.wait()==0, '上のログを確認してください'


In [ ]:
from IPython.display import display, Image
report=json.loads((work/'result.json').read_text())
print(json.dumps(report,ensure_ascii=False,indent=2))
display(Image(filename=str(work/'trajectory.png')))
import py3Dmol
v=py3Dmol.view(width=850,height=500)
# Cαを残基順に対応づけ、剛体重ね合わせする。
import numpy as np
ref_lines=[l for l in (work/'1TEN.pdb').read_text().splitlines() if l.startswith('ATOM') and l[21]=='A']
pred_lines=(work/'design.pdb').read_text().splitlines()
def ca(lines):
    return np.array([[float(l[30:38]),float(l[38:46]),float(l[46:54])] for l in lines if l.startswith('ATOM') and l[12:16].strip()=='CA' and l[16] in [' ','A']])
a=ca(pred_lines);b=ca(ref_lines)
assert a.shape==b.shape, '残基対応を確認してください'
u,_,vt=np.linalg.svd((a-a.mean(0)).T@(b-b.mean(0)))
d=np.eye(3);d[-1,-1]=np.sign(np.linalg.det(u@vt));rot=u@d@vt
aligned=(a-a.mean(0))@rot+b.mean(0)
saved_rmsd=float(np.sqrt(np.mean(np.sum((aligned-b)**2,axis=1))))
print('保存した構造のCα RMSD [Å]:',saved_rmsd)
report=json.loads((work/'result.json').read_text());report['saved_structure_ca_rmsd_A']=saved_rmsd
report['metrics_note']='last_step_metricsは最終反復、保存構造と配列は保存対象の最良候補。RMSDは保存PDBから計算。'
(work/'result.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
out=[]
for l in pred_lines:
    if l.startswith('ATOM'):
        q=(np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])])-a.mean(0))@rot+b.mean(0)
        l=l[:30]+''.join(f'{v:8.3f}' for v in q)+l[54:]
    out.append(l)
(work/'aligned.pdb').write_text(chr(10).join(out))
v.addModel(chr(10).join(ref_lines),'pdb');v.setStyle({'model':0},{'cartoon':{'color':'gray','opacity':0.45}})
v.addModel(chr(10).join(out),'pdb');v.setStyle({'model':1},{'cartoon':{'color':'spectrum'}})
v.zoomTo();v.show()
print('灰色：標的、虹色：設計中の予測構造')


## 読み方
- 損失の推移と、最後の離散配列を確認。
- 目標と予測構造のRMSDを読む。原子の対応と重ね合わせの条件が重要です。
- pLDDTが高くても独立な検証ではありません。AfDesignはこの予測器を通して設計しています。
- 反復数を増やす場合は、計算時間も記録します。研究で使う場合は複数seed・モデル・実験での確認が必要です。


In [ ]:
import zipfile
with zipfile.ZipFile('bioinfo07_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in ['1TEN.pdb','design.pdb','design.fasta','result.json','trajectory.png','run_design.py','aligned.pdb']:z.write(f)
from google.colab import files
files.download('bioinfo07_results.zip')


## 再利用元
公式[ColabDesign AfDesign](https://github.com/sokrypton/ColabDesign/blob/e31a56fe1d9b4de25c8697f3a28b75892941cc72/af/design.ipynb)。授業ではfixbbだけを使用し、依存環境と反復数を固定しています。

Frank et al. (2024), Scalable protein design using optimization in a relaxed sequence space. https://doi.org/10.1126/science.adq1741

ColabDesignのコードと学習済みパラメータは元のライセンスに従います。
